## Operational Flow 


In [1]:
from sqlglot import parse_one, exp
import uuid
from owlready2 import get_ontology, sync_reasoner

### Step-2 : AST (`sqlgpt_parser`) to AST Tree conversion

In [2]:
# --- Type Mapper ---

CLAUSE_TYPES = {
    "From": "FromClause",
    "Group": "GroupByClause",
    "Into": "IntoClause",
    "Limit": "LimitClause",
    "Select": "SelectClause",
    "Set": "SetClause",
    "Update": "UpdateClause",
    "Values": "ValuesClause",
    "Where": "WhereClause",
    "Order": "OrderByClause",
    "Having": "HavingClause",
    "Join": "JoinClause"
}

STATEMENT_TYPES = {
    "Delete": "DeleteStatement",
    "Insert": "InsertStatement",
    "Update": "UpdateStatement",
    "Select": "SelectStatement"
}

EXPRESSION_CATEGORIES = {
    "Alias": "Alias",
    "Table": "TableRef",
    "Column": "ColumnRef",
    "Wildcard": "WildCard",
    "Identifier": "Identifier",
    # Functions
    "Sum": "Function",
    "Count": "Function",
    "Avg": "Function",
    "Max": "Function",
    "Min": "Function",
    "Concat": "Function",
    "Coalesce": "Function",
    # Operators
    "And": "Operator",
    "Or": "Operator",
    "EQ": "Operator",
    "GT": "Operator",
    "LT": "Operator",
    "Add": "Operator",
    "Sub": "Operator",
    "Mul": "Operator",
    "Div": "Operator",
    # Literals
    "Literal": "Literal"
} # TODO: Add more expression types as needed

def map_node_type(node, statement=False): # TODO: look the logic is complete but needs to be tested
    """
    Map a sqlglot AST node to its corresponding type in the ontology.
    """
    class_name = type(node).__name__
    if statement:
        if class_name in STATEMENT_TYPES:
            return "Statement", STATEMENT_TYPES[class_name]
        else:
            return "Statement", class_name
    else:
        if class_name in CLAUSE_TYPES:
            return "Clause", CLAUSE_TYPES[class_name]
        elif class_name in EXPRESSION_CATEGORIES: # TODO: I need to set the specific category for each expression type in ast object
            return "Expression", EXPRESSION_CATEGORIES[class_name]
        else:
            return "Expression", class_name


In [3]:

# --- TreeNode Class ---

class TreeNode:
    """ 
    TreeNode represents a node in the SQL AST tree. It can be a statement, clause, or expression.
    """
    def __init__(self, sqlglot_node, parent=None):
        self.remove = False
        self.parent = parent
        self.children = []
        self.sqlglot_node = sqlglot_node
        self.kind, self.name = map_node_type(sqlglot_node, statement=True if parent is None else False)

        self.id = "n"+str(uuid.uuid4())[:6]
        sqlglot_node.meta['id'] = self.id

        if self.name == "ColumnRef":
            self.refcol = None
            self.reftable = None
            # self.reference_id = None
            parts = sqlglot_node.parts
            if len(parts) == 2:
                self.reftable = parts[0].this
                self.refcol = parts[1].this
            elif len(parts) == 1:
                self.refcol = parts[0].this

        if self.name == "TableRef":
            self.reftable = sqlglot_node.name
            if sqlglot_node.alias:
                self.refalias = sqlglot_node.alias
            # self.reference_id = None

        if self.name == "Alias":
            alias_expr = sqlglot_node.args.get("alias")
            if isinstance(alias_expr, exp.Identifier):
                self.alias = alias_expr.name

        if self.name == "Literal":
            self.value = sqlglot_node.this
        
        if self.name == "Operator":
            if (type(sqlglot_node).__name__ in ["And", "Or"]):
                self.logics = True 
            else:
                self.logics = False

    def add_child(self, child):
        self.children.append(child)

    def __repr__(self):
        suffix = f" ({self.name})"
        if self.name == "ColumnRef":
            suffix += f" [reftable={self.reftable}, refcol={self.refcol}]"
        if self.name == "TableRef":
            suffix += f" [reftable={self.reftable}]"
        if self.name == "Alias" and hasattr(self, "alias"):
            suffix += f" [name={self.alias}]"
        if self.name == "Literal" and hasattr(self, "value"):
            suffix += f" [value={self.value}]"

        if hasattr(self, "reference_id"):
            suffix += f" [reference_id={self.reference_id}]"
        if hasattr(self, "remove"):
            suffix += f" [remove={self.remove}]"
        if hasattr(self, "refalias"):
            suffix += f" [refalias={self.refalias}]"
        if hasattr(self, "logics"):
            suffix += f" [logics={self.logics}]"
        return f"<({self.id}) {self.kind} | {suffix}>"

    def walk(self):
        yield self
        for child in self.children:
            yield from child.walk()

    def print_tree(self, level=0):
        print("  " * level + repr(self))
        for child in self.children:
            child.print_tree(level + 1)


In [4]:
# --- Tree Builder ---

def build_tree(sqlglot_node, parent=None):

    if parent is not None:
        clause_root = TreeNode(sqlglot_node, parent=parent)
        root_node = parent
        root_node.add_child(clause_root)
    else:
        root_node = TreeNode(sqlglot_node)
        clause_root = TreeNode(sqlglot_node, parent=root_node)
        root_node.add_child(clause_root)
    

    found_first_clause = False
    for key, child in sqlglot_node.args.items():
        if child is None:
            continue

        children = child if isinstance(child, list) else [child]

        for expr in children:
            if not isinstance(expr, exp.Expression): # did not understand the logic of the check
                continue

            c_kind, c_name = map_node_type(expr)

            if not found_first_clause and c_kind != "Clause":
                sub = _build_recursive(expr, clause_root)
                if sub:
                    clause_root.add_child(sub)
            else:
                found_first_clause = True
                clause_node = TreeNode(expr, parent=root_node)
                root_node.add_child(clause_node)

                for _, grandchild in expr.args.items():
                    if isinstance(grandchild, exp.Expression):
                        if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier): # this is a weak check, so try to transfer this is recursion
                            continue  # Skip Identifier under Table
                        child_node = _build_recursive(grandchild, clause_node)
                        if child_node:
                            clause_node.add_child(child_node)
                    elif isinstance(grandchild, list):
                        for item in grandchild:
                            if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                continue  # Skip Identifier under Table
                            if isinstance(item, exp.Expression):
                                child_node = _build_recursive(item, clause_node)
                                if child_node:
                                    clause_node.add_child(child_node)
    if parent is None:
        return root_node


def _build_recursive(sqlglot_node, parent_node):
    if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "ColumnRef":
        return None

    if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "TableRef":
        return None
    
    if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "Alias":
        return None

    if isinstance(sqlglot_node, exp.TableAlias) and parent_node.name == "TableRef":
        return None

    if isinstance(sqlglot_node, exp.Paren):
        # return first child of the paren expression to avoid unnecessary nesting
        first_child = next(iter(sqlglot_node.args.values()))
        return _build_recursive(first_child, parent_node)

    if isinstance(sqlglot_node, exp.Select):
        current = build_tree(sqlglot_node, parent_node)
    else:
        current = TreeNode(sqlglot_node, parent_node)
        for _, child in sqlglot_node.args.items():
            if isinstance(child, exp.Expression):
                if isinstance(sqlglot_node, exp.Table) and isinstance(child, exp.Identifier):
                    continue
                child_node = _build_recursive(child, current)
                if child_node:
                    current.add_child(child_node)
            elif isinstance(child, list):
                for item in child:
                    if isinstance(sqlglot_node, exp.Table) and isinstance(item, exp.Identifier):
                        continue
                    if isinstance(item, exp.Expression):
                        child_node = _build_recursive(item, current)
                        if child_node:
                            current.add_child(child_node)

    return current


In [5]:
def get_node_children(node):
    """Helper to get all child expression nodes from a node's args."""
    for value in node.args.values():
        if isinstance(value, exp.Expression):
            yield value
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, exp.Expression):
                    yield item

def pretty_print_ast(node, prefix="", is_last=True):
    """
    Recursively prints a formatted tree of the AST, including our custom IDs.
    """
    # Use the first 8 characters of the UUID for cleaner printing
    short_id = node.meta.get('id', 'N/A')[:8]
    
    # Get the node's type and any direct value (like a column name)
    node_name = node.__class__.__name__
    if hasattr(node, 'this'):
        node_name += f" ({node.this})"

    # Print the current node with a tree-like structure
    connector = "└── " if is_last else "├── "
    print(f"{prefix}{connector}{node_name}  [ID: {short_id}]")

    # Recurse for children
    children = list(get_node_children(node))
    for i, child in enumerate(children):
        new_prefix = prefix + ("    " if is_last else "│   ")
        pretty_print_ast(child, prefix=new_prefix, is_last=(i == len(children) - 1))

### Step-3 : AST Tree to Ontology Instance 

### Ontology reasoning

In [6]:
class OntologyInstantiator:
    def __init__(self, onto_path):
        """
        Initializes the instantiator by loading the ontology.
        """
        self.onto = get_ontology(onto_path).load()
        self.class_lookup = {cls.name: cls for cls in self.onto.classes()}
        
        # Tracking lists are now instance attributes
        self.table_ref_instances = []
        self.column_ref_instances = []

    def get_table_ref_instances(self):
        """
        Returns the list of instantiated TableRef objects.
        """
        return {inst.name: "Aligned" if getattr(inst, 'hasAlignmentState', []) == [] else "Violated" for inst in self.table_ref_instances }
    
    def get_column_ref_instances(self):
        """
        Returns the list of instantiated ColumnRef objects.
        """
        
        return {inst.name: "Aligned" if getattr(inst, 'hasAlignmentState', []) == [] else "Violated" for inst in self.column_ref_instances }

    def walk_and_instantiate(self, node, agent_id, parent_instance=None):
        """
        Recursively walks a tree and instantiates ontology classes.
        All dependencies like 'self.onto' and 'self.class_lookup' are explicit.
        """
        node_class = self.class_lookup.get(node.name)
        if not node_class:
            print(f"[Warning] Missing class in ontology: {node.name}")
            return None

        inst = node_class(node.id)

        # Attach parent link
        if parent_instance:
            inst.immediateParentNode.append(parent_instance)

        if node.kind == "Statement":
            # Link Statement to Agent via agent_id
            ref_obj = self.onto.search_one(iri="*"+agent_id)
            if ref_obj:
                print(f"[Debug] Linking {node.name} instance {inst.name} to Agent {ref_obj.name}.")
                inst.executedBy.append(ref_obj)
            else:
                print(f"[Warning] Agent {agent_id} not found in ontology.")

        # Link TableRef to ontology Table via reference_id
        if node.name == "TableRef":
            self.table_ref_instances.append(inst)
            if hasattr(node, "reference_id"):
                ref_obj = self.onto.search_one(iri="*" + node.reference_id)
                if ref_obj:
                    inst.referencesToTable.append(ref_obj)
                else:
                    print(f"[Warning] TableRef unresolved: {node.reference_id}")

        # Link ColumnRef to ontology Column via reference_id
        if node.name == "ColumnRef":
            self.column_ref_instances.append(inst)
            if hasattr(node, "reference_id"):
                ref_obj = self.onto.search_one(iri="*" + node.reference_id)
                if ref_obj:
                    inst.referencesToColumn.append(ref_obj)
                else:
                    print(f"[Warning] ColumnRef unresolved: {node.reference_id}")

        # Recurse
        for child in node.children:
            self.walk_and_instantiate(child, agent_id, parent_instance=inst)

        return inst

    def process_tree(self, tree_root, agent_id):
        """
        Main method to process the tree, run the reasoner, and save the result.
        """
        with self.onto:
            # Assume resolve_references_with_ontology is defined elsewhere
            # and now takes the ontology as an argument if needed.
            # resolve_references_with_ontology(tree_root, self.onto) 
            
            self.walk_and_instantiate(tree_root, agent_id)
            sync_reasoner(infer_property_values=True, debug=0)

    def resolve_references_with_ontology(self, root_node):

        table_entities = {}
        for t in self.onto.individuals():
            # if len(t.is_a) == 2 and "Table" == t.is_a[1].name:
            # print(f"[Debug] Processing individual: {t.name}")
            if "Table" in [cls.name for cls in t.is_a if type(cls).__name__ == "ThingClass"]:
                table_entities[t.TableName[0]] = t.name
        # print(table_entities)
        
        column_lookup = {}
        for n in self.onto.individuals():
            # if "Column" == n.is_a[0].name:
            if "Column" in [cls.name for cls in n.is_a if type(cls).__name__ == "ThingClass"]:
                # print("Here")
                for tbl in getattr(n, "columnOfTable", []):
                    table_name = tbl.TableName[0]
                    column_lookup[(table_name, n.ColumnName[0])] = n.name
        # print(column_lookup)

        def resolve_for_statement(stmt_node):
            alias_map = {}  # alias -> table name
            tables_in_scope = []  # actual tables in FROM clause

            for child in stmt_node.children:
                if child.name == "FromClause":
                    for t in child.walk():
                        if t.name == "TableRef" and hasattr(t, "reftable"):
                            table_name = t.reftable
                            alias_map[table_name] = table_name
                            tables_in_scope.append(table_name)

                            if hasattr(t, "refAlias"):
                                alias_map[t.refAlias] = table_name

            for node in stmt_node.walk():
                if node.name == "TableRef" and hasattr(node, "reftable"):
                    tbl_key = node.reftable
                    ref_table = table_entities.get(tbl_key)
                    if ref_table:
                        node.reference_id = table_entities[node.reftable]

                elif node.name == "ColumnRef" and hasattr(node, "refcol"):
                    # Use declared reftable if exists
                    if hasattr(node, "reftable") and node.reftable:
                        tbl_key = alias_map.get(node.reftable, node.reftable)
                        col_key = (tbl_key, node.refcol)
                    else:
                        if len(tables_in_scope) == 1:
                            tbl_key = tables_in_scope[0]
                            # print(f"[Info] Using single table '{tbl_key}' for column '{node.refcol}'.")
                            col_key = (tbl_key, node.refcol)
                        else:
                            print(f"[Warning] Ambiguous column '{node.refcol}' with no reftable in multi-table query.")
                            continue
                    

                    ref_col = column_lookup.get(col_key)
                    # print(f"[Debug] Resolving ColumnRef {ref_col} with key {col_key}.")
                    if ref_col:
                        node.reference_id = ref_col

            for node in root_node.walk():
                if node.kind == "Statement":
                    resolve_for_statement(node)
        
        # return root_node

    
    def save(self, output_path, file_format="rdfxml"):
        """
        Saves the modified ontology to a file.
        """
        self.onto.save(file=output_path, format=file_format)

In [7]:
class ASTPruner:
    """
    A robust class to parse a SQL query and prune it based on a set of
    invalid node IDs, working directly with the sqlglot AST.
    """
    def __init__(self, ast):        
        # Parse and decorate the AST upon initialization

        self.decorated_ast = ast

    def _decorate_ast_with_ids(self, ast_node):
        """Adds a unique ID to a 'meta' attribute on each node."""
        for node in ast_node.walk():
            if not hasattr(node, 'meta'):
                node.meta = {}
            node.meta['id'] = str(uuid.uuid4())
        return ast_node

    def _pruning_transformer(self, node):
        """
        The core logic passed to ast.transform(). It prunes the tree
        bottom-up based on the user's rules.
        """
        # Base Case: If the node itself is explicitly violated, remove it.
        if hasattr(node, 'meta') and node.meta.get('id') in self.violated_ids:
            print(f"--> Removing violated node {node.this if hasattr(node, 'this') else node.__class__.__name__} [ID: {node.meta['id']}]")
            return None # Returning None deletes the node

        # Recursive Cases: Check if a parent node has become invalid because
        # its essential children were removed by previous transformations.

        # Rule: If a Function's argument is removed, remove the function.
        # (e.g., SUM(invalid_column) -> SUM(None) -> remove SUM)
        if isinstance(node, exp.Func) and node.this is None:
            print(f"--> Cascading removal of Function '{node.sql()}' because its argument was removed.")
            return None

        # Rule: If a binary Operator's operands are removed, remove the operator.
        # (e.g., invalid_column = 5 -> None = 5 -> remove =)
        if isinstance(node, exp.Binary) and (node.left is None or node.right is None):
            print(f"--> Cascading removal of Operator '{node.sql()}' because an operand was removed.")
            return None

        # Rule: If a JOIN's 'ON' clause becomes invalid/empty, remove the JOIN.
        if isinstance(node, exp.Join) and node.on is None:
            print(f"--> Cascading removal of JOIN '{node.sql()}' because its ON clause was removed.")
            return None

        # Final check: For a SELECT statement, if all selected columns are removed,
        # it's an invalid statement. We return the node but it will be empty.
        if isinstance(node, exp.Select) and not node.expressions:
            print(f"--> All columns removed from a SELECT clause.")

        # If no rules match, keep the node.
        return node

    def prune(self, violdated_ids):
        """
        Runs the pruning process on the AST.
        
        Returns:
            The modified AST object.
        """
        self.violated_ids = violdated_ids
        
        if not self.decorated_ast:
            return None
        
        # ast.transform() will recursively apply our transformer function
        pruned_ast = self.decorated_ast.transform(self._pruning_transformer)
        return pruned_ast

    def get_node_by_id(self, node_id):
        """Helper to find a node by its ID."""
        return self.decorated_ast.find(exp.Expression, finder=lambda n: hasattr(n, 'meta') and n.meta.get('id') == node_id)

    def pretty_print(self, ast_node=None):
        """Prints the tree structure of the given AST or the instance's AST."""
        # A simplified printer for demonstration
        if ast_node is None:
            ast_node = self.decorated_ast
        for node in ast_node.walk():
            indent = "  " * node.depth
            node_id = node.meta.get('id', 'N/A')[:8]
            print(f"{indent}├── {node.__class__.__name__} ({node.sql()}) [ID: {node_id}]")

### AST to sql

---
## Run

In [8]:
query = "SELECT bank, SUM(amount), type AS total FROM Transaction WHERE (account_id = 123) and (amount > 100) GROUP BY account_id"
# query = "SELECT account_id, account_to, bank_to, amount FROM Order;"

# query = """
# SELECT
#     c.customer_id,
#     c.name,
#     (
#         SELECT COUNT(*)
#         FROM orders o
#         WHERE o.customer_id = c.customer_id
#     ) AS total_orders,
#     (
#         SELECT AVG(o2.total_amount)
#         FROM orders o2
#         WHERE o2.customer_id = c.customer_id
#           AND o2.order_date > CURRENT_DATE - INTERVAL '30 days'
#     ) AS avg_recent_order_amount
# FROM customers c
# WHERE c.status = 'active';

# """

# query = "SELECT account_id, account_to, bank_to, amount FROM Order;"

In [9]:
ast = parse_one(query)
tree_root = build_tree(ast)

# Print the tree
tree_root.print_tree()

<(nd60cf0) Statement |  (SelectStatement) [remove=False]>
  <(n56ad43) Clause |  (SelectClause) [remove=False]>
    <(n252488) Expression |  (ColumnRef) [reftable=None, refcol=bank] [remove=False]>
    <(n67d42d) Expression |  (Function) [remove=False]>
      <(n1127dd) Expression |  (ColumnRef) [reftable=None, refcol=amount] [remove=False]>
    <(n4ba5e9) Expression |  (Alias) [name=total] [remove=False]>
      <(nd6e11b) Expression |  (ColumnRef) [reftable=None, refcol=type] [remove=False]>
  <(n7e3dba) Clause |  (FromClause) [remove=False]>
    <(n818b9b) Expression |  (TableRef) [reftable=Transaction] [remove=False]>
  <(nb0d547) Clause |  (WhereClause) [remove=False]>
    <(n2239a9) Expression |  (Operator) [remove=False] [logics=True]>
      <(n32c47e) Expression |  (Operator) [remove=False] [logics=False]>
        <(n58d0d9) Expression |  (ColumnRef) [reftable=None, refcol=account_id] [remove=False]>
        <(nf393ae) Expression |  (Literal) [value=123] [remove=False]>
      <(

In [10]:
pretty_print_ast(ast)

└── Select (None)  [ID: n56ad43]
    ├── Column (bank)  [ID: n252488]
    │   └── Identifier (bank)  [ID: N/A]
    ├── Sum (amount)  [ID: n67d42d]
    │   └── Column (amount)  [ID: n1127dd]
    │       └── Identifier (amount)  [ID: N/A]
    ├── Alias (type)  [ID: n4ba5e9]
    │   ├── Column (type)  [ID: nd6e11b]
    │   │   └── Identifier (type)  [ID: N/A]
    │   └── Identifier (total)  [ID: N/A]
    ├── From (Transaction)  [ID: n7e3dba]
    │   └── Table (Transaction)  [ID: n818b9b]
    │       └── Identifier (Transaction)  [ID: N/A]
    ├── Where ((account_id = 123) AND (amount > 100))  [ID: nb0d547]
    │   └── And ((account_id = 123))  [ID: n2239a9]
    │       ├── Paren (account_id = 123)  [ID: N/A]
    │       │   └── EQ (account_id)  [ID: n32c47e]
    │       │       ├── Column (account_id)  [ID: n58d0d9]
    │       │       │   └── Identifier (account_id)  [ID: N/A]
    │       │       └── Literal (123)  [ID: nf393ae]
    │       └── Paren (amount > 100)  [ID: N/A]
    │      

In [11]:
onto_path = "../ontology_file/aput.rdf"
output_path = "../ontology_file/resolved_query_instances.rdf"
agent_id = "a0012"

# 1. Create an instance of the class
instantiator = OntologyInstantiator(onto_path)

# 2. Resolve references in the tree using the ontology
instantiator.resolve_references_with_ontology(tree_root)

# 2. Process the tree
instantiator.process_tree(tree_root, agent_id=agent_id)

# 3. Save the results
instantiator.save(output_path)

print("Ontology processing complete and saved.")

[Debug] Linking SelectStatement instance nd60cf0 to Agent a0012.
Ontology processing complete and saved.


In [12]:
table_ref_instances = instantiator.get_table_ref_instances()
column_ref_instances = instantiator.get_column_ref_instances()

for key, value in table_ref_instances.items():
    print(f"TableRef {key} → Alignment: {value}")

for key, value in column_ref_instances.items():
    print(f"ColumnRef {key} → Alignment: {value}")

TableRef n818b9b → Alignment: Aligned
ColumnRef n252488 → Alignment: Aligned
ColumnRef n1127dd → Alignment: Aligned
ColumnRef nd6e11b → Alignment: Aligned
ColumnRef n58d0d9 → Alignment: Aligned
ColumnRef ne41cd0 → Alignment: Aligned
ColumnRef n1731e5 → Alignment: Aligned


In [13]:
violated_ids = [col_ref for col_ref in column_ref_instances if column_ref_instances[col_ref] == "Violated"]  + [table_ref for table_ref in table_ref_instances if table_ref_instances[table_ref] == "Violated"]
pruner = ASTPruner(ast)
pruned_ast = pruner.prune(violated_ids)
# To inspect the pruned AST
pruner.pretty_print(pruned_ast)

├── Select (SELECT bank, SUM(amount), type AS total FROM Transaction WHERE (account_id = 123) AND (amount > 100) GROUP BY account_id) [ID: n56ad43]
  ├── Column (bank) [ID: n252488]
  ├── Sum (SUM(amount)) [ID: n67d42d]
  ├── Alias (type AS total) [ID: n4ba5e9]
  ├── From (FROM Transaction) [ID: n7e3dba]
  ├── Where (WHERE (account_id = 123) AND (amount > 100)) [ID: nb0d547]
  ├── Group (GROUP BY account_id) [ID: n8f8ae5]
    ├── Identifier (bank) [ID: N/A]
    ├── Column (amount) [ID: n1127dd]
    ├── Column (type) [ID: nd6e11b]
    ├── Identifier (total) [ID: N/A]
    ├── Table (Transaction) [ID: n818b9b]
    ├── And ((account_id = 123) AND (amount > 100)) [ID: n2239a9]
    ├── Column (account_id) [ID: n1731e5]
      ├── Identifier (amount) [ID: N/A]
      ├── Identifier (type) [ID: N/A]
      ├── Identifier (Transaction) [ID: N/A]
      ├── Paren ((account_id = 123)) [ID: N/A]
      ├── Paren ((amount > 100)) [ID: N/A]
      ├── Identifier (account_id) [ID: N/A]
        ├── EQ (acco

In [14]:
pruned_ast.sql()

'SELECT bank, SUM(amount), type AS total FROM Transaction WHERE (account_id = 123) AND (amount > 100) GROUP BY account_id'